# Corpus audit, deduplication, and split checks

**Purpose:** Present committed corpus audit results for byte matching, deduplication, near-duplicate removal, and group-aware evaluation splits.

**Inputs (committed result/provenance files read):**
- [`data/audit/audit_summary.json`](../data/audit/audit_summary.json)
- [`data/audit/exact_duplicates.parquet`](../data/audit/exact_duplicates.parquet)
- [`data/audit/near_duplicates.parquet`](../data/audit/near_duplicates.parquet)
- [`data/audit/cross_repo_overlap.parquet`](../data/audit/cross_repo_overlap.parquet)

**Outputs:** Rendered audit summaries and optional preview tables in this notebook only; no files are written.

**Expected runtime:** ~5 seconds (loads committed CSV/JSON/parquet files only).

**Traceability links:** [`FINAL_REPORT.md`](../results/reports/FINAL_REPORT.md) and [`CLAIM_LEDGER.md`](../publication/CLAIM_LEDGER.md).

**Execution notes:** Analysis-only notebook. It resolves `ROOT` by walking upward to `plan.md`, uses no absolute paths or machine identifiers, does not download corpora, does not retrain tokenizers, and has no hidden state. Randomness: none used. Exploratory analysis: none; cells below directly present committed result files.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / 'plan.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

def rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def note_missing(path):
    print(f'NOTE: optional input is missing or empty: {path}. The notebook continues without it.')

def load_csv(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    return pd.read_csv(p)

def load_parquet(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    try:
        return pd.read_parquet(p)
    except Exception as exc:
        print(f'NOTE: could not read optional parquet {path}: {exc}')
        return pd.DataFrame()

def load_json(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return {}
    return json.loads(p.read_text())

def load_text(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return ''
    return p.read_text()

def safe_read_json_path(path):
    try:
        return json.loads(Path(path).read_text())
    except Exception as exc:
        print(f'NOTE: could not read {rel(path)}: {exc}')
        return {}

print(f'Repository root: {rel(ROOT)}')


Repository root: .


## Corpus audit

Authoritative file: [`audit_summary.json`](../data/audit/audit_summary.json). Optional detail tables are linked above and previewed only when present. No corpora are downloaded.

In [2]:
audit = load_json('data/audit/audit_summary.json')
exact_dups = load_parquet('data/audit/exact_duplicates.parquet')
near_dups = load_parquet('data/audit/near_duplicates.parquet')
cross_overlap = load_parquet('data/audit/cross_repo_overlap.parquet')

In [3]:
if audit:
    rows = [
        ('byte_match_ok', audit.get('byte_match_ok')),
        ('byte_match_spread', audit.get('byte_match_spread')),
        ('corpora_content_bytes.azure', audit.get('corpora_content_bytes', {}).get('azure')),
        ('corpora_content_bytes.general', audit.get('corpora_content_bytes', {}).get('general')),
        ('corpora_content_bytes.mixed', audit.get('corpora_content_bytes', {}).get('mixed')),
        ('azure_exact_removed', audit.get('exact_dedup_azure', {}).get('removed_exact')),
        ('general_exact_removed', audit.get('exact_dedup_general', {}).get('removed_exact')),
        ('cross_corpus_removed_near_train', audit.get('cross_corpus_near_dup_eval', {}).get('removed_near_train')),
        ('leakage_violations', len(audit.get('leakage_violations', []))),
    ]
    display(pd.DataFrame(rows, columns=['audit_check', 'value']))
else:
    note_missing('data/audit/audit_summary.json')

,audit_check,value
0,byte_match_ok,True
1,byte_match_spread,0
2,corpora_content_bytes.azure,33554432
3,corpora_content_bytes.general,33554432
4,corpora_content_bytes.mixed,33554432
5,azure_exact_removed,2
6,general_exact_removed,134
7,cross_corpus_removed_near_train,9
8,leakage_violations,0


In [4]:
if audit:
    print(json.dumps(audit.get('azure_splits', {}), indent=2))

{
  "azure_cross_service_holdout": 965,
  "azure_same_service_holdout": 488,
  "azure_train": 3799,
  "n_cross_service_holdout_services": 5,
  "n_leakage_groups": 20,
  "n_services": 20
}


In [5]:
for label, df in [('exact_duplicates', exact_dups), ('near_duplicates', near_dups), ('cross_repo_overlap', cross_overlap)]:
    print(f'{label}: {len(df):,} rows' if not df.empty else f'{label}: optional detail table missing or empty')
    if not df.empty:
        display(df.head(10))

exact_duplicates: 97 rows


,sha256,n,document_ids,repos
0,bcd9056b89a5af05f29496e72d2e6979568263fd888187...,2,sha256:fc58c4e636494096bfa55fb206ff6dc915706bc...,MicrosoftDocs/azure-devops-docs
1,075b7c18f9e714616842be31e10969aab422c980dd59ed...,2,sha256:337e14d4b21bbbc822add18f95daa38913b1cae...,MicrosoftDocs/azure-devops-docs
2,1ba328cd29b13870024433cda7f54e254ea25b9706abf1...,2,sha256:847b4f20ccce6f8b52b6865f8d0387ba236b011...,MicrosoftDocs/azure-devops-docs
3,e90e3507556e5b50007d109eeb32233e51287aef29f78a...,2,sha256:1b5965c329403cf44a8234b0c1ea04154dedefb...,MicrosoftDocs/azure-docs
4,3dc13df7b13b469659b06c5c498b2008db876cb31743ad...,2,sha256:da858d85a476464f53bf6a16e7a8e559b51fcb5...,MicrosoftDocs/azure-docs
5,e3ca55219d32613f1aedd4fd9a616595db14ebd2888c75...,3,sha256:c5935bcf1b9013c863684f2ab7cd7b0bc19f330...,Salesforce/wikitext
6,539760afe7f58070392fc44e381a606217635411df5813...,2,sha256:875cc6cc936223a70fcd7357c756dacf3ec9fc4...,Salesforce/wikitext
7,e5fca74f5a6219971c52f0bdea4c04bc8a226f65f5af37...,4,sha256:7b21bb41d3f87540e56268e14b75a3d10718d4b...,Salesforce/wikitext
8,4d1102a9b54215351f1ef308564f17a1d3535359a0186b...,3,sha256:6e340c9c78a27341c066d54172edb43f61be68b...,Salesforce/wikitext
9,a911091bd2755230f10d9b7ce47f1fa353e0df0a472216...,2,sha256:07bc1dfba781aaf17d5a4bdfaf29ee0c2061bc7...,Salesforce/wikitext


near_duplicates: 163 rows


,id_a,id_b,jaccard
0,sha256:11b7bc39704b64b226bb5dcc7647ad6eee89fb9...,sha256:fddc31596f57273e160b494c66deb942fa3e3e9...,0.828125
1,sha256:666cc412c7d1717ed1c17436f124aa334e26dfe...,sha256:96106a5e19623f7d0551d47ff8947c0a598750c...,0.921875
2,sha256:c7065a50d977513b4cac5b9d9934d881058b91b...,sha256:b4b438474a2b421fbb626e5117d68923b91d47e...,0.890625
3,sha256:5112b2e0a73debe8c3e9ceaa424158bb835cbd3...,sha256:0a16fbd86b9bd7d8ce6672b30581350f85d4c08...,0.859375
4,sha256:b2dd570811b9d4f09eb802a1f8307fe2c222fc2...,sha256:e8cc794a547e702d0b09c4ebfb3e4adb9fcf50e...,0.851562
5,sha256:9d2bf2ca88f0a4eb1a565183c3062dfbbfaf7ed...,sha256:e25e766d3f115ca0bd5b29c9437936281d5f64e...,0.859375
6,sha256:9d2bf2ca88f0a4eb1a565183c3062dfbbfaf7ed...,sha256:1d5abde9e6707c7a685f7e4bc391806891ccad5...,0.851562
7,sha256:5071604c36133c896a50978c2fa7c2bff287b59...,sha256:091e8adc80fcfa4d6bc4ed8d191348afe57b1f5...,0.867188
8,sha256:1d5abde9e6707c7a685f7e4bc391806891ccad5...,sha256:58aede5113e9b6b0b8dd3127b4ef58643bbc1a4...,0.843750
9,sha256:c92e4f4e4867377f7d10828fbb740057fcb482c...,sha256:00e350bac28f1027ec67690ee2da7d501908afe...,0.820312


cross_repo_overlap: 1 rows


,sha256,repos,n_repos
0,,,0
